# **TensorFlow**
Тема: Класифікація рукописних цифр (MNIST) із простою багатошаровою нейронною мережею
Мета: Ознайомитися з основними концепціями TensorFlow (тензори, шари, оптимізатор, функція втрат) на прикладі простої задачі класифікації зображень.

**Завдання**  (те, що дано і треба розібратись)

Навчання моделі на датасеті MNIST

Завантажте MNIST (використавши вбудовані методи tensorflow.keras.datasets або інші доступні джерела).

Створіть просту нейронну мережу (мінімум один прихований шар, наприклад, Dense(128, activation='relu')) у tf.keras.Sequential.
Оцінка моделі (evaluate)

Виведіть точність (accuracy) на тестовій вибірці за допомогою model.evaluate(...).

Побудуйте Confusion Matrix, скориставшись пакетом sklearn.metrics.confusion_matrix.

Виведіть Classification Report (precision, recall, f1-score) через sklearn.metrics.classification_report.

Візуалізація прикладів

Виведіть щонайменше 5 зображень із тестового набору (matplotlib.pyplot.imshow(...)).

Для кожного зображення вкажіть передбачений клас (argmax) та істинну мітку (y_test).

Обговоріть, чи трапляються помилки, і як вони виглядають.

**Додаткові експерименти** **(те, що треба виконати)**

Змініть кількість шарів або змінні параметри (наприклад, додайте ще один Dense чи збільште кількість епох).

Збережіть або завантажте модель (model.save("mnist_model.h5"), model.load_model(...)) і перевірте, чи результат зберігається.

Оцініть, як це впливає на Confusion Matrix, Classification Report та точність.

**Оформлення результатів** **(те, що треба виконати)**

У звіті:
Опишіть архітектуру моделі (кількість шарів, кількість нейронів).

Наведіть (скриншоти чи вивід) Confusion Matrix та Classification Report.

Покажіть декілька прикладів зображень із правильним або помилковим передбаченням.

Підготуйте **висновки**: на яких цифрах модель помиляється найчастіше, яка загальна точність, чи покращились результати після змін у мережі тощо.


## Практична частина

Нижче наведено реалізацію всіх пунктів завдання: базова модель, оцінювання, візуалізація результатів, додатковий експеримент та перевірка збереження моделі.


In [ ]:
# -*- coding: utf-8 -*-

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import datasets, models, layers
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def build_model(hidden_layers, dropout_rate=0.0):
    model = models.Sequential()
    model.add(tf.keras.Input(shape=(28, 28)))
    model.add(layers.Flatten())

    for units in hidden_layers:
        model.add(layers.Dense(units, activation='relu'))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(10, activation='softmax'))
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def evaluate_and_plot(model, model_name, x_test, y_test):
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    predictions = model.predict(x_test, verbose=0)
    y_pred = np.argmax(predictions, axis=1)

    print(f"{model_name} - test loss: {test_loss:.4f}")
    print(f"{model_name} - test accuracy: {test_acc:.4f}")
    print(f"\n{model_name} - Classification Report:\n")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.show()

    return {
        'loss': test_loss,
        'accuracy': test_acc,
        'y_pred': y_pred,
        'report': classification_report(y_test, y_pred, output_dict=True)
    }


def show_examples(x_data, y_true, y_pred, title, count=5, only_errors=False):
    if only_errors:
        indices = np.where(y_true != y_pred)[0][:count]
    else:
        indices = np.arange(count)

    plt.figure(figsize=(14, 4))
    for i, idx in enumerate(indices):
        plt.subplot(1, len(indices), i + 1)
        plt.imshow(x_data[idx], cmap='gray')
        plt.title(f"Pred: {y_pred[idx]}\nTrue: {y_true[idx]}")
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# 1. Завантаження даних MNIST
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()

# 2. Нормалізація
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)


# 3. Базова модель
baseline_model = build_model(hidden_layers=[128])
baseline_model.summary()

baseline_history = baseline_model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)


# 4. Оцінка базової моделі
baseline_results = evaluate_and_plot(baseline_model, 'Baseline model', x_test, y_test)


# 5. Візуалізація прикладів
show_examples(x_test, y_test, baseline_results['y_pred'], 'Перші 5 передбачень базової моделі', count=5)
show_examples(x_test, y_test, baseline_results['y_pred'], 'Приклади помилок базової моделі', count=5, only_errors=True)


# 6. Додатковий експеримент: більше шарів + більше епох
experiment_model = build_model(hidden_layers=[256, 128], dropout_rate=0.2)
experiment_model.summary()

experiment_history = experiment_model.fit(
    x_train,
    y_train,
    epochs=8,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

experiment_results = evaluate_and_plot(experiment_model, 'Experiment model', x_test, y_test)
show_examples(x_test, y_test, experiment_results['y_pred'], 'Приклади помилок покращеної моделі', count=5, only_errors=True)


# 7. Збереження та завантаження моделі
experiment_model.save('mnist_model.h5')
loaded_model = tf.keras.models.load_model('mnist_model.h5')
loaded_loss, loaded_acc = loaded_model.evaluate(x_test, y_test, verbose=0)

print(f"\nLoaded model - test loss: {loaded_loss:.4f}")
print(f"Loaded model - test accuracy: {loaded_acc:.4f}")
print(f"Accuracy difference after reload: {abs(loaded_acc - experiment_results['accuracy']):.8f}")


# 8. Порівняння результатів
comparison_df = pd.DataFrame([
    {
        'model': 'Baseline',
        'architecture': 'Flatten -> Dense(128) -> Dense(10)',
        'epochs': 5,
        'test_accuracy': baseline_results['accuracy'],
        'macro_f1': baseline_results['report']['macro avg']['f1-score']
    },
    {
        'model': 'Experiment',
        'architecture': 'Flatten -> Dense(256) -> Dropout(0.2) -> Dense(128) -> Dropout(0.2) -> Dense(10)',
        'epochs': 8,
        'test_accuracy': experiment_results['accuracy'],
        'macro_f1': experiment_results['report']['macro avg']['f1-score']
    },
    {
        'model': 'Loaded experiment model',
        'architecture': 'Loaded from mnist_model.h5',
        'epochs': 8,
        'test_accuracy': loaded_acc,
        'macro_f1': np.nan
    }
])

comparison_df.round(4)
